In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# Loading data

In [2]:
learn_data = pd.read_csv("preprocess_train_v4.csv", header = None)
learn_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt',
       'LogSgot', 'Target']
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot,Target
0,48,0.511111,0.226640,4.615385,1.504077,5.641907,2.564949,4.304065,0
1,39,0.473684,-0.182383,3.115942,0.641854,5.192957,3.737670,4.127134,0
2,23,0.300000,-0.264120,3.100000,0.000000,5.356586,3.713572,4.382027,0
3,42,0.285714,-0.374875,3.018868,-0.356675,5.023881,3.555348,4.394449,0
4,54,0.504425,0.604032,4.250000,3.117950,6.324359,3.401197,3.610918,0


In [3]:
learn_data.isna().value_counts()

Age    DBRatio  ALBIScore  Glob   LogTB  LogAlkphos  LogSgpt  LogSgot  Target
False  False    False      False  False  False       False    False    False     450
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state = 42)

# Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

crossval_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])
validation_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# The classifiers, by themselves

### QDA

In [6]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 100
m = 10
regs = np.logspace(start = -4, stop = -0.5, num = n)
priors = [(p / m, 1 - p / m) for p in range(1, m)]

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs,
                                        'QDA__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X_train, y_train)
QDA_search.best_params_

{'QDA__priors': (0.8, 0.19999999999999996),
 'QDA__reg_param': 0.31622776601683794}

In [7]:
QDA_search.best_score_

0.642161680613336

In [8]:
qda_priors = QDA_search.best_params_['QDA__priors']
qda_reg_param = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = qda_priors,
                                          reg_param = qda_reg_param)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["QDA", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.642162,0.658896,0.641275,0.691667


In [9]:
qda_pipeline.fit(X_train, y_train)
validation_df.loc["QDA", :] = compute_metrics(y_val, qda_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.65,0.652911,0.647895,0.688889


In [10]:
confusion(y_val, qda_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	16	13
	0	15	46
Accuracy: 68.89%


## Logistic Regression

In [11]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression())])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs,
                                           'logreg__class_weight' : weights},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X_train, y_train)
logreg_search.best_params_

{'logreg__C': 0.01747528400007685, 'logreg__class_weight': {0: 0.3, 1: 0.7}}

In [12]:
logreg_search.best_score_

0.6698802363872829

In [13]:
logreg_C = logreg_search.best_params_["logreg__C"]
logreg_weights = logreg_search.best_params_["logreg__class_weight"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = logreg_weights)
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["LogReg-Best", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
QDA,0.642162,0.658896,0.641275,0.691667


In [14]:
logreg_pipeline.fit(X_train, y_train)
validation_df.loc["LogReg-Best", :] = compute_metrics(y_val, logreg_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889


In [15]:
confusion(y_val, logreg_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	22	7
	0	22	39
Accuracy: 67.78%


## Sigmoid kernel SVC

In [16]:
sigsvc = SVC(kernel = "sigmoid", class_weight = "balanced")
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)
# gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]

sigsvc_search = GridSearchCV(estimator = sigsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__class_weight' : weights},
#                                            'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
sigsvc_search.fit(X_train, y_train)
sigsvc_search.best_params_

# Results from before:
# {'svc__C': 21.209508879201902, 'svc__gamma': 3.236329950002764e-05}

{'svc__C': 1.3219411484660293, 'svc__class_weight': {0: 0.4, 1: 0.6}}

In [17]:
sigsvc_search.best_score_

0.6562064915223207

In [18]:
sigsvc_C = sigsvc_search.best_params_['svc__C']
sigsvc_weights = sigsvc_search.best_params_['svc__class_weight']
# sigsvc_gamma = sigsvc_search.best_params_['svc__gamma']
sigsvc_best = SVC(kernel = "sigmoid",
                  C = sigsvc_C,
                  gamma = "scale",
                  class_weight = sigsvc_weights,
                  probability = True)
sigsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", sigsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(sigsvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Sigmoid SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667


In [19]:
sigsvc_pipeline.fit(X_train, y_train)
validation_df.loc["Sigmoid SVC", :] = compute_metrics(y_val, sigsvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556


In [20]:
confusion(y_val, sigsvc_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	16	13
	0	18	43
Accuracy: 65.56%


## Polynomial SVC

In [21]:
polysvc = SVC(kernel = "poly", class_weight = "balanced")
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc)])

n = 100
m = 10
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]
Cs = np.logspace(start = -1, stop = 2, num = n)
# gammas = np.logspace(start = -2, stop = 2, num = m) / X.shape[0]
degrees = [2, 3]

polysvc_search = GridSearchCV(estimator = polysvc_pipeline,
                              param_grid = {'svc__C' : Cs,
                                            'svc__class_weight' : weights,
#                                             'svc__gamma' : gammas,
                                            'svc__degree' : degrees},
                              scoring = 'f1_macro',
                              cv = 5)
polysvc_search.fit(X_train, y_train)
polysvc_search.best_params_

# Results from before:
# {'svc__C': 0.35564803062231287,
#  'svc__degree': 3,
#  'svc__gamma': 0.2222222222222222}

{'svc__C': 4.977023564332111,
 'svc__class_weight': {0: 0.3, 1: 0.7},
 'svc__degree': 3}

In [22]:
polysvc_search.best_score_

0.6372899791988085

In [23]:
polysvc_C = polysvc_search.best_params_['svc__C']
# polysvc_gamma = polysvc_search.best_params_['svc__gamma']
polysvc_degree = polysvc_search.best_params_['svc__degree']
polysvc_weights = polysvc_search.best_params_['svc__class_weight']
polysvc_best = SVC(kernel = "poly",
                   C = polysvc_C,
                   gamma = "scale",
                   degree = polysvc_degree,
                   class_weight = polysvc_weights,
                   probability = True)
polysvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", polysvc_best)])

cross_val_results = pd.DataFrame(cross_validate(polysvc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Polynomial SVC", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556


In [24]:
polysvc_pipeline.fit(X_train, y_train)
validation_df.loc["Polynomial SVC", :] = compute_metrics(y_val, polysvc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556


In [25]:
confusion(y_val, polysvc_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	23	6
	0	22	39
Accuracy: 68.89%


## Random Forest

In [26]:
from imblearn.pipeline import Pipeline as PipelineIMB
from imblearn.under_sampling import RandomUnderSampler

rf_pipeline = Pipeline([('scaler', StandardScaler()),
                        ('rf', RandomForestClassifier(random_state = 777))])

m = 10
criteria = ["gini", "entropy", "log_loss"]
max_features = ["sqrt", "log2", None]
depths = [2, 4, 6, 8]
min_samples = [5, 10, 20, 30]
weights = [{0 : p / m, 1 : 1 - p / m} for p in range(1, m)]

rf_search = GridSearchCV(estimator = rf_pipeline,
                         param_grid = {'rf__criterion' : criteria,
                                      'rf__max_features' : max_features,
                                      'rf__max_depth' : depths,
                                      'rf__min_samples_split' : min_samples,
                                      'rf__class_weight' : weights},
                         scoring = 'f1_macro',
                         cv = 5)
rf_search.fit(X_train, y_train)
rf_search.best_params_

{'rf__class_weight': {0: 0.3, 1: 0.7},
 'rf__criterion': 'gini',
 'rf__max_depth': 8,
 'rf__max_features': 'sqrt',
 'rf__min_samples_split': 10}

In [31]:
rf_search.best_score_

0.6634502526905754

In [41]:
rf_criterion = rf_search.best_params_['rf__criterion']
rf_max_depth = rf_search.best_params_['rf__max_depth']
rf_max_features = rf_search.best_params_['rf__max_features']
rf_min_samples_split = rf_search.best_params_['rf__min_samples_split']
rf_weights = rf_search.best_params_['rf__class_weight']
rf_best = RandomForestClassifier(criterion = rf_criterion,
                                max_depth = rf_max_depth,
                                max_features = rf_max_features,
                                min_samples_split = rf_min_samples_split,
                                n_estimators = 100,
                                class_weight = rf_weights,
                                random_state = 777)
rf_pipeline = PipelineIMB([("scaler", StandardScaler()),
                           ('rf', rf_best)])

cross_val_results = pd.DataFrame(cross_validate(rf_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Random Forest", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Random Forest,0.66345,0.667249,0.665489,0.730556
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556


In [42]:
rf_pipeline.fit(X_train, y_train)
validation_df.loc["Random Forest", :] = compute_metrics(y_val, rf_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Random Forest,0.637097,0.634822,0.640212,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556


In [43]:
confusion(y_val, rf_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	14	15
	0	13	48
Accuracy: 68.89%


# Voting classifiers

The classifiers, by themselves, have similar performance. However, when predicting the training data they show to have different opinions. A consensus should be established.

In [44]:
qda_pipeline.fit(X_train, y_train)
qda_labels = qda_pipeline.predict(X_val)
logreg_pipeline.fit(X_train, y_train)
logreg_labels = logreg_pipeline.predict(X_val)
sigsvc_pipeline.fit(X_train, y_train)
sigsvc_labels = sigsvc_pipeline.predict(X_val)
polysvc_pipeline.fit(X_train, y_train)
polysvc_labels = polysvc_pipeline.predict(X_val)
rf_pipeline.fit(X_train, y_train)
rf_labels = rf_pipeline.predict(X_val)

In [45]:
pd.Series(np.logical_and(np.logical_and(logreg_labels == qda_labels,
                                        logreg_labels == sigsvc_labels,
                                        logreg_labels == rf_labels),
                         logreg_labels == polysvc_labels)).value_counts()

True     67
False    23
Name: count, dtype: int64

Let's make them vote, then.

In [54]:
estimators = [("qda", qda_pipeline),
              ("logreg", logreg_pipeline),
              ("sigsvc", sigsvc_pipeline),
              ("rf", rf_pipeline)]
votingclass = VotingClassifier(estimators = estimators, voting = "hard")
votingclass.fit(X_train, y_train)
validation_df.loc["Voting", :] = compute_metrics(y_val, votingclass.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
LogReg-Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Voting,0.65,0.652911,0.647895,0.688889
Random Forest,0.637097,0.634822,0.640212,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556


In [55]:
n = 10
weights = [(p1 / n, p2 / n, p3 / n, 1 - (p1 + p2 + p3) / n)
           for p1 in range(n + 1)
           for p2 in range(n + 1 - p1)
           for p3 in range(n + 1 - p1 - p2)]

vote_search = GridSearchCV(estimator = votingclass,
                           param_grid = {'weights' : weights},
                           cv = 5,
                           scoring = "f1_macro")
vote_search.fit(X_train, y_train)
vote_search.best_params_

{'weights': (0.0, 0.2, 0.4, 0.4)}

In [56]:
vote_search.best_score_

0.6874191368380715

In [85]:
estimators_hard = [("logreg", logreg_pipeline),
                   ("sigsvc", sigsvc_pipeline),
                   ("rf", rf_pipeline)]
vote_weights = vote_search.best_params_['weights']
vote_hard_best = VotingClassifier(estimators = estimators,
                                  voting = "hard")

cross_val_results = pd.DataFrame(cross_validate(vote_hard_best, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Voting Hard Best", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Voting Hard Best,0.674847,0.688318,0.671707,0.725
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Voting Soft Best,0.66988,0.721632,0.678516,0.691667
Random Forest,0.66345,0.667249,0.665489,0.730556
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556


In [91]:
estimators = [("logreg", logreg_pipeline),
              ("sigsvc", sigsvc_pipeline),
              ("polysvc", polysvc_pipeline),
              ("rf", rf_pipeline)]
votingclass = VotingClassifier(estimators = estimators, voting = "hard")

n = 10
weights = [(p1 / n, p2 / n, p3 / n, 1 - (p1 + p2 + p3) / n)
           for p1 in range(n + 1)
           for p2 in range(n + 1 - p1)
           for p3 in range(n + 1 - p1 - p2)]

vote_search = GridSearchCV(estimator = votingclass,
                           param_grid = {'weights' : weights},
                           cv = 5,
                           scoring = "f1_macro")
vote_search.fit(X_train, y_train)
vote_search.best_params_

{'weights': (0.1, 0.4, 0.1, 0.4)}

In [92]:
vote_search.best_score_

0.6976165498802821

In [99]:
estimators_hard = [("logreg", logreg_pipeline),
                   ("sigsvc", sigsvc_pipeline),
                   ("polysvc", polysvc_pipeline),
                   ("rf", rf_pipeline)]
vote_weights = vote_search.best_params_['weights']
vote_hard_best = VotingClassifier(estimators = estimators,
                                  voting = "hard",
                                  weights = vote_weights)

cross_val_results = pd.DataFrame(cross_validate(vote_hard_best, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Voting Hard Best", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Voting Hard Best,0.66988,0.721632,0.678516,0.691667
Random Forest,0.66345,0.667249,0.665489,0.730556
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556
Voting Soft Best,0.630193,0.628524,0.642102,0.719444


In [100]:
vote_hard_best.fit(X_train, y_train)
validation_df.loc["Voting Hard Best", :] = compute_metrics(y_val, vote_hard_best.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
LogReg-Best,0.665856,0.698982,0.673913,0.677778
Voting Hard Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Voting,0.65,0.652911,0.647895,0.688889
Random Forest,0.637097,0.634822,0.640212,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556
Voting SoftBest,0.57672,0.574901,0.581538,0.644444


In [102]:
confusion(y_val, vote_hard_best.predict(X_val))

		Predicted
		+1	0
Real	+1	22	7
	0	22	39
Accuracy: 67.78%


In [96]:
estimators = [("logreg", logreg_pipeline),
              ("sigsvc", sigsvc_pipeline),
              ("polysvc", polysvc_pipeline),
              ("rf", rf_pipeline)]
votingclass = VotingClassifier(estimators = estimators, voting = "soft")

n = 10
weights = [(p1 / n, p2 / n, p3 / n, 1 - (p1 + p2 + p3) / n)
           for p1 in range(n + 1)
           for p2 in range(n + 1 - p1)
           for p3 in range(n + 1 - p1 - p2)]

vote_search = GridSearchCV(estimator = votingclass,
                           param_grid = {'weights' : weights},
                           cv = 5,
                           scoring = "f1_macro")
vote_search.fit(X_train, y_train)
vote_search.best_params_

{'weights': (1.0, 0.0, 0.0, 0.0)}

In [97]:
vote_search.best_score_

0.6698802363872829

### Making predictions

In [63]:
test_data = pd.read_csv("preprocess_test_v4.csv", header = None)
test_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot']
test_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot
0,11,0.142857,-0.460075,3.000000,-0.356675,6.383507,3.258097,3.367296
1,62,0.500000,-0.172320,5.000000,0.587787,5.411646,4.234107,5.043425
2,60,0.285714,-0.460075,3.818182,-0.356675,5.159055,3.465736,2.639057
3,60,0.491228,0.226237,4.102564,1.740466,5.365976,6.021023,6.745236
4,48,0.222222,-0.260240,3.000000,-0.105361,5.164786,3.178054,3.988984


In [82]:
test_y = pd.read_csv("test_y.csv")['Label']

In [84]:
rf_pipeline.fit(X, y)

labels_rf = pd.DataFrame(columns = ['ID', 'Label'])
labels_rf['Label'] = pd.DataFrame(rf_pipeline.predict(test_data))
labels_rf['ID'] = labels_rf.index + 1
labels_rf.to_csv('new_predictions/rf_best_fs.csv', index = False)
compute_metrics(test_y, labels_rf['Label'])

[0.7138519924098672,
 0.7064622124863089,
 0.7241379310344828,
 0.7758620689655172]

In [101]:
vote_hard_best.fit(X_train, y_train)

labels_vote = pd.DataFrame(columns = ['ID', 'Label'])
labels_vote['Label'] = pd.DataFrame(vote_hard_best.predict(test_data))
labels_vote['ID'] = labels_vote.index + 1
labels_vote.to_csv('new_predictions/vote_hard_best_fs.csv', index = False)
compute_metrics(test_y, labels_vote['Label'])

[0.7092261664106089, 0.7493610806863819, 0.706969696969697, 0.7327586206896551]

# Bagging Classifier

In [111]:
from sklearn.ensemble import BaggingClassifier

bagger = BaggingClassifier(estimator = logreg_pipeline, max_samples = 0.7, n_estimators = 100)
bag_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("bagger", bagger)])

cross_val_results = pd.DataFrame(cross_validate(bagger, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["Bagging - Log.Reg", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Voting Hard Best,0.66988,0.721632,0.678516,0.691667
Bagging - PolySVC,0.663558,0.703171,0.667342,0.691667
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556
Voting Soft Best,0.630193,0.628524,0.642102,0.719444
Random Forest,0.629689,0.634605,0.629298,0.697222


# AdaBoost Classifier

In [116]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth = 3)
adaboost = AdaBoostClassifier(estimator = tree, n_estimators = 100)
boost_pipeline = Pipeline([("scaler", StandardScaler()),
                           ("boost", adaboost)])

cross_val_results = pd.DataFrame(cross_validate(boost_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["AdaBoost", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Voting Hard Best,0.66988,0.721632,0.678516,0.691667
Bagging - PolySVC,0.663558,0.703171,0.667342,0.691667
AdaBoost,0.656363,0.649262,0.67458,0.747222
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556
Extra Trees,0.634801,0.638225,0.633327,0.702778
Random Forest,0.632293,0.637755,0.631461,0.697222
Voting Soft Best,0.630193,0.628524,0.642102,0.719444


In [118]:
n = 20
m = 10

learning_rates = np.logspace(start = -2, stop = 1, num = n)
n_estimators = [5, 10, 20, 40, 70, 100]

ada_search = GridSearchCV(estimator = boost_pipeline,
                          param_grid = {"boost__n_estimators" : n_estimators,
                                        "boost__learning_rate" : learning_rates},
                          cv = 5,
                          scoring = "f1_macro")
ada_search.fit(X_train, y_train)
ada_search.best_params_

{'boost__learning_rate': 1.1288378916846884, 'boost__n_estimators': 70}

In [119]:
ada_search.best_score_

0.6325802932355249

In [123]:
tree = DecisionTreeClassifier(max_depth = 2)
adaboost = AdaBoostClassifier(estimator = tree, n_estimators = 100)

ada_search = GridSearchCV(estimator = adaboost,
                          param_grid = {"n_estimators" : n_estimators,
                                        "learning_rate" : learning_rates},
                          cv = 5,
                          scoring = "f1_macro")
ada_search.fit(X_train, y_train)
ada_search.best_params_

{'learning_rate': 1.623776739188721, 'n_estimators': 100}

In [124]:
ada_search.best_score_

0.7002202026343131

In [125]:
tree = DecisionTreeClassifier(max_depth = 2)
adaboost = AdaBoostClassifier(estimator = tree, n_estimators = 100, learning_rate = 1.624)

cross_val_results = pd.DataFrame(cross_validate(adaboost, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["AdaBoost - Depth 2", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
AdaBoost - Depth 2,0.697646,0.690614,0.710366,0.769444
LogReg-Best,0.66988,0.721632,0.678516,0.691667
Voting Hard Best,0.66988,0.721632,0.678516,0.691667
Bagging - PolySVC,0.663558,0.703171,0.667342,0.691667
AdaBoost,0.656363,0.649262,0.67458,0.747222
Sigmoid SVC,0.656206,0.678062,0.653898,0.7
QDA,0.642162,0.658896,0.641275,0.691667
Polynomial SVC,0.63729,0.696596,0.657983,0.655556
Extra Trees,0.634801,0.638225,0.633327,0.702778
Random Forest,0.632293,0.637755,0.631461,0.697222


In [130]:
adaboost = AdaBoostClassifier(estimator = tree, n_estimators = 100, learning_rate = 1.624)
adaboost.fit(X_train, y_train)
validation_df.loc["AdaBoost - Depth 2", :] = compute_metrics(y_val, adaboost.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
Polynomial SVC,0.678735,0.716224,0.688889,0.688889
LogReg-Best,0.665856,0.698982,0.673913,0.677778
Voting Hard Best,0.665856,0.698982,0.673913,0.677778
QDA,0.65,0.652911,0.647895,0.688889
Voting,0.65,0.652911,0.647895,0.688889
Random Forest,0.637097,0.634822,0.640212,0.688889
Sigmoid SVC,0.62149,0.628321,0.619223,0.655556
AdaBoost - Depth 2,0.6,0.597795,0.646115,0.7
Voting SoftBest,0.57672,0.574901,0.581538,0.644444
